# Reproduce the leak: a search with dates vs a time-locked search

This notebook runs one case from the repository's benchmark (F3, a petrol and diesel price query with a cutoff in May 2026) **twice** with **your own** Tavily key:

1. **No dates:** a plain search.
2. **Time-locked:** `timelocked_search` (start and end date, `filter_by_published_date`, never past the cutoff).

For each result it shows the URL, Tavily's `published_date` and whether that date is after the cutoff. It costs at most 4 Tavily credits per run (two basic searches, usually 2); the notebook prints the credits Tavily reports.

**What the table means.** A "yes" is an *after-cutoff dated result (date flag)*. Confirmed leaks need the Wayback check in the repository (`shadow_cpi/leakcheck.py`, `tools/wayback_retry.py`). Today's index may differ from our run, so the counts here may differ from the ones in the README; nothing below is a hard-coded expected count.

In [ ]:
# 1. Install from the GitHub repository (standard library only, nothing else is installed).
!git clone --depth 1 https://github.com/bhavikjain026/shadow-cpi.git
!pip install -q ./shadow-cpi

In [ ]:
# 2. Your own Tavily key. getpass hides what you type; the key is kept in memory only, never printed or saved.
from getpass import getpass
from timelock import TavilyHTTP

client = TavilyHTTP(api_key=getpass("Your Tavily API key: "))

In [ ]:
# 3. The F3 case: query, cutoff and request settings are read from the benchmark file, not typed here.
import json
from pathlib import Path

CASES = next(p for p in (Path("shadow-cpi/benchmarks/timelock34/cases.jsonl"), Path("benchmarks/timelock34/cases.jsonl"))
             if p.exists())
case = next(c for c in map(json.loads, CASES.read_text(encoding="utf-8").splitlines()) if c["id"] == "F3")
QUERY, CUTOFF = case["question"], case["cutoff"]
SKIP = {"query", "include_usage", "start_date", "end_date", "filter_by_published_date"}
plain = {k: v for k, v in case["requests"]["no_dates"].items() if k not in SKIP}
locked = {k: v for k, v in case["requests"]["dates_filter"].items() if k not in SKIP}
MAX_CREDITS, ESTIMATE = 4, 2                     # two basic searches at 1 credit each
assert ESTIMATE <= MAX_CREDITS
print(f"case {case['id']}: {QUERY!r}, cutoff {CUTOFF}; estimated {ESTIMATE} credits, cap {MAX_CREDITS}")

In [ ]:
# 4. Run both searches.
from timelock import timelocked_search

no_dates = client.search(QUERY, **plain)
timelocked = timelocked_search(client, QUERY, CUTOFF, start_date=case["requests"]["dates_filter"]["start_date"], **locked)
credits = sum(((r.get("usage") or {}).get("credits") or 0) for r in (no_dates, timelocked.response))
print(f"Tavily request ids: {no_dates.get('request_id')} (no dates), {timelocked.request_id} (time-locked)")
print(f"Tavily credits reported for this run: {credits} (cap {MAX_CREDITS})")

In [ ]:
# 5. Per result: URL, Tavily published_date, after the cutoff?
from timelock import iso_date

def rows(results):
    out = []
    for r in results:
        d = iso_date(r.get("published_date")) or None
        out.append((r["url"], d, "no date" if d is None else ("yes" if d > CUTOFF else "no")))
    return out

arms = {"no dates": rows(no_dates.get("results") or []), "time-locked": rows(timelocked)}
for name, table in arms.items():
    print(f"\n== {name} ==  (after-cutoff dated results: date flags, not confirmed leaks)")
    for url, d, after in table:
        print(f"  {after:>7}  {d or '-':<10}  {url[:100]}")

In [ ]:
# 6. Summary, computed from the run above.
for name, table in arms.items():
    n_after = sum(a == "yes" for _, _, a in table)
    print(f"{name}: {n_after} of {len(table)} results are dated after the cutoff {CUTOFF} ({sum(a == 'no date' for _, _, a in table)} undated).")
print("These are after-cutoff dated results (date flags); confirmed leaks need the Wayback check in the repo. Today's index may differ from our run.")